# UHVDB r6 assembly-only clusters

Genus and family clusters whose member sequences are all `db_type == "Assembly"` in the released UHVDB r6 metadata (`uhvdb_metadata.tsv.gz`).

A cluster is assembly-only when every row in that cluster is Assembly. The percent is that count divided by all clusters at the rank.

In [3]:
from pathlib import Path

import polars as pl

METADATA_PATH = Path(
    "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB"
    "/toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz"
)

df = pl.read_csv(
    METADATA_PATH,
    separator="\t",
    columns=["db_type", "species_cluster_id", "genus_cluster_id", "family_cluster_id"],
)

print(f"Loaded {df.height:,} rows from {METADATA_PATH}")
print(df.group_by("db_type").len().sort("len", descending=True))

for rank, col in [
    ("species", "species_cluster_id"),
    ("genera", "genus_cluster_id"),
    ("family", "family_cluster_id"),
]:
    n = df.filter(pl.col(col).is_not_null()).select(pl.col(col).n_unique()).item()
    print(f"UHVDB r6 {rank} clusters: {n:,}")

Loaded 1,541,381 rows from /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz
shape: (2, 2)
┌──────────┬─────────┐
│ db_type  ┆ len     │
│ ---      ┆ ---     │
│ str      ┆ u32     │
╞══════════╪═════════╡
│ Database ┆ 1287973 │
│ Assembly ┆ 253408  │
└──────────┴─────────┘
UHVDB r6 species clusters: 291,605
UHVDB r6 genera clusters: 53,506
UHVDB r6 family clusters: 1,590


In [4]:
def assembly_only_clusters(cluster_col: str) -> tuple[int, int]:
    grouped = (
        df.filter(pl.col(cluster_col).is_not_null())
        .group_by(cluster_col)
        .agg(
            pl.len().alias("n_sequences"),
            (pl.col("db_type") == "Assembly").sum().alias("n_assembly"),
        )
    )
    n_only = grouped.filter(pl.col("n_sequences") == pl.col("n_assembly")).height
    return n_only, grouped.height


for rank, col in [("genus", "genus_cluster_id"), ("family", "family_cluster_id")]:
    n_only, n_total = assembly_only_clusters(col)
    pct = 100 * n_only / n_total
    print(f"{rank} clusters with only Assembly viruses: {n_only:,} / {n_total:,} ({pct:.2f}%)")

genus clusters with only Assembly viruses: 5,331 / 53,506 (9.96%)
family clusters with only Assembly viruses: 101 / 1,590 (6.35%)
